    
    
# 🧠 Programación Dinámica (PD): qué es y cuándo usarla

**Idea central:** resolver un problema grande descomponiéndolo en **subproblemas** cuyos resultados **se reutilizan**.
Dos propiedades clave:

* **Subestructura óptima** ⚙️: la solución óptima del problema se compone de soluciones óptimas de subproblemas.
* **Subproblemas superpuestos** 🔁: el árbol de recursión repite los mismos subproblemas una y otra vez.

**Memoization (top-down)** = *recursión + caché*. Guardas el resultado de cada subproblema la primera vez que lo calculas y, si vuelve a aparecer, lo lees de la caché en O(1).

> **Regla de oro de complejidad en PD**:
> **Tiempo ≈ #estados distintos × costo por estado**
> **Espacio ≈ #estados almacenados + profundidad del call stack**

---

# 🧭 Memoization vs. Tabulation

* **Memoization (Top-Down)** 🧗‍♂️

  * Empiezas por el problema completo y desciendes recursivamente.
  * Calculas *sólo* los estados que realmente se necesitan.
  * Más natural cuando ya tienes una recursión clara.
  * Cuidado con la **profundidad de pila** y **mutabilidad de claves**.

* **Tabulation (Bottom-Up)** 🧱

  * Llenas una tabla iterativamente desde casos base.
  * Evitas recursión y controlas mejor memoria/orden.
  * Requiere pensar el **orden de llenado**.

> En clase: empieza con **memoization** (más intuitivo), y luego muestra cómo convertirlo a **tabulation**.

---

# 🧩 Anatomía de una solución con memoization

1. **Define el estado** 🎯: ¿qué parámetros hacen único al subproblema? (índices, capacidad, posición, etc.)
2. **Escribe la recursión “de fuerza bruta”** ✍️: sin preocuparte por eficiencia (claro y correcto).
3. **Casos base** 🪜: condiciones de frontera que cortan la recursión.
4. **Transición** 🔀: cómo combinas subsoluciones (min, max, suma, conteo, etc.).
5. **Caché** 🗃️: diccionario o `@lru_cache`. Claves **inmutables** (tuplas, enteros, strings).
6. **Complejidad** 📏: cuenta estados (tamaño del dominio del estado) y el costo por estado.
7. **Validación** ✅: prueba con ejemplos pequeños; activa/inspecciona la caché si dudas.

---

# 🧠 Modelo mental (checklist de diseño)

* 🔎 **¿Hay recomputación?** Dibuja un árbol de recursión pequeño y busca subllamadas repetidas.
* 🧱 **Estado mínimo**: “si dos llamadas tienen el mismo conjunto de hechos relevantes, deben compartir estado”.
* ⛳ **Base cases claros**: que no dependan de la caché; evita ciclos.
* 🧮 **Transición limpia**: que cada camino de decisión reduzca el problema (convergencia).
* 🧭 **Clave de caché**: usa tuplas de parámetros inmutables; **no** metas listas/objetos mutables.
* 🧯 **Efectos laterales**: memoization es para funciones *puras* (mismo input → mismo output).
* 📦 **Memoria**: ¿cuántos estados caben? Si el dominio es grande, evalúa *tabulation* con compresión de estados.

---

# 🧪 Plantilla universal (Python, con y sin `lru_cache`)

### Con `functools.lru_cache`

```python
from functools import lru_cache

@lru_cache(maxsize=None)
def dp(estado1: int, estado2: int) -> int:
    # 1) casos base
    if condicion_de_base:
        return valor_base
    # 2) transiciones (subllamadas)
    opcionA = dp(nuevo_estadoA1, nuevo_estadoA2)
    opcionB = dp(nuevo_estadoB1, nuevo_estadoB2)
    # 3) combinación (min/max/suma)
    return min(opcionA, opcionB)  # o max / suma / etc.
```

### Con diccionario manual

```python
from typing import Dict, Tuple

def solve(param1: int, param2: int) -> int:
    memo: Dict[Tuple[int, int], int] = {}

    def dp(a: int, b: int) -> int:
        # caché
        if (a, b) in memo:
            return memo[(a, b)]

        # casos base
        if condicion_de_base:
            memo[(a, b)] = valor_base
            return valor_base

        # transiciones
        res = ... # combina dp(subestado...) según la recurrencia
        memo[(a, b)] = res
        return res

    return dp(param1, param2)
```

👉 **Explicación rápida:**

* Elegimos **clave** como tupla de parámetros del subproblema.
* Guardamos el resultado antes de retornar.
* Así, **cada estado se calcula una sola vez**.

---

# 🎓 Ejemplos guiados

## 1) Fibonacci (ilustra superposición de subproblemas)

**Estado:** `f(n)`
**Base:** `f(0)=0`, `f(1)=1`
**Transición:** `f(n)=f(n-1)+f(n-2)`
**Complejidad:** tiempo O(n), espacio O(n) con memo (O(n) caché + O(n) call stack)

```python
from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n: int) -> int:
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)
```

**Por qué funciona:** sin memo hay 2^n llamadas; con memo cada `n` se calcula una vez.

---

## 2) LCS – Longest Common Subsequence (longitud)

**Problema:** longitud de la subsecuencia común más larga entre `s` y `t`.
**Estado:** `(i, j)` = longitud de LCS entre `s[i:]` y `t[j:]`.
**Base:** si `i==len(s)` o `j==len(t)`, LCS = 0.
**Transición:**

* Si `s[i]==t[j]`: `1 + LCS(i+1, j+1)`
* Si no: `max(LCS(i+1, j), LCS(i, j+1))`
  **#estados:** `O(len(s)*len(t))`.
  **Complejidad:** tiempo O(n·m), espacio O(n·m) (caché) + O(n+m) (pila).

```python
from functools import lru_cache

def lcs_len(s: str, t: str) -> int:
    n, m = len(s), len(t)

    @lru_cache(maxsize=None)
    def dp(i: int, j: int) -> int:
        if i == n or j == m:
            return 0
        if s[i] == t[j]:
            return 1 + dp(i+1, j+1)
        return max(dp(i+1, j), dp(i, j+1))

    return dp(0, 0)
```

**Claves didácticas:** muestra cómo el árbol de recursión se “aplana” a una malla `i×j`.

---

## 3) Caminos en una grilla con obstáculos (conteo)

**Problema:** dado un grid `grid` de 0/1, contar caminos de `(0,0)` a `(n-1,m-1)` moviendo solo derecha/abajo sin pisar celdas con `1`.
**Estado:** `(i, j)` = #caminos desde `(i, j)` a la meta.
**Base:** si `(i, j)` es la meta → 1; si está fuera o hay obstáculo → 0.
**Transición:** `dp(i, j) = dp(i+1, j) + dp(i, j+1)`
**#estados:** `O(n·m)`
**Complejidad:** tiempo O(n·m), espacio O(n·m) + O(n+m).

```python
from functools import lru_cache
from typing import List

def count_paths(grid: List[List[int]]) -> int:
    n, m = len(grid), len(grid[0])

    @lru_cache(maxsize=None)
    def dp(i: int, j: int) -> int:
        if i >= n or j >= m or grid[i][j] == 1:
            return 0
        if i == n-1 and j == m-1:
            return 1
        return dp(i+1, j) + dp(i, j+1)

    return dp(0, 0)
```

**Observación:** si cambias la suma por `min`/`max` + costos, obtienes la variante de **camino mínimo**.

---

# 🚩 Errores comunes y cómo evitarlos

* **Claves mutables en la caché** ❌ → usa **tuplas**/enteros/strings.
* **Olvidar casos base** → bucles infinitos o recursión profunda.
* **Estados incompletos** → colisiones en la caché (resultados incorrectos).
* **Side effects** en funciones memoizadas → resultados incoherentes.
* **Reinicializar la caché** en cada llamada externa inadvertidamente.
* **Profundidad de recursión** en inputs grandes → evaluar *tabulation* o `sys.setrecursionlimit` con cuidado.

---

# 🧮 Cómo analizar complejidad (plantilla)

1. **Cuenta estados**: producto/cartesiano de los rangos de cada parámetro del estado.

   * Ej.: LCS → `|i|∈[0..n]`, `|j|∈[0..m]` ⇒ `O(n·m)` estados.
2. **Costo por estado**: cuántas transiciones evalúas y su costo.

   * Suele ser O(1)–O(grado de opciones).
3. **Tiempo total**: `#estados × costo_por_estado`.
4. **Espacio**: memoria de caché + call stack (profundidad máxima).

---

# 🛠️ Convertir a Bottom-Up (cuando conviene)

* Define **orden topológico** natural (por tamaños crecientes, índices decrecientes, etc.).
* **Reusa memoria** (rolling arrays) si la transición solo usa filas/columnas previas.
* Evita **desbordes de pila** y mejora **localidad de caché** de CPU.

*Ej.: LCS bottom-up llena una tabla `n+1 × m+1` desde el fondo; se puede comprimir a 2 filas si solo lees la fila siguiente.*

---

# 📚 Mini-banco de ejercicios intro (perfectos para memoization)

1. **Stairs / Climbing**: #formas de subir `n` escalones con pasos 1 o 2.
2. **Decode Ways**: cuántas decodificaciones para un string de dígitos.
3. **Coin Change** (conteo y/o mínimo #monedas).
4. **LCS / Edit Distance** (par de strings).
5. **Grid Paths / Min Path Sum** con obstáculos.

---




# 🎒 El problema de la mochila (Knapsack 0/1)

Tienes una mochila que aguanta máximo **5 kg**.  
Hay **4 objetos disponibles**:

| Objeto | Peso | Valor |
|---|---|---|
| Guitarra 🎸 | 1 kg | \$1500 |
| Laptop 💻 | 3 kg | \$2000 |
| Libro 📚 | 2 kg | \$500 |
| Cámara 📷 | 2 kg | \$1000 |

## ❓ Pregunta

¿Qué objetos debes meter en la mochila para maximizar el valor total sin pasarte de **5 kg**?

In [ ]:
def dp_mochila(peso_maximo, pesos, valores):
    matriz = [[0]*(peso_maximo + 1) for _ in range(len(pesos) + 1)]
    for i in range(1, len(pesos) + 1):
        for j in range(1, peso_maximo + 1):
            if pesos[i-1] > j:
                print(pesos[i-1], j)
                print(matriz[i][j],"arriba", matriz[i-1][j])
                matriz[i][j] = matriz[i-1][j]
            else:
                matriz[i][j] = max(matriz[i-1][j], valores[i-1] + matriz[i-1][j - pesos[i-1]])
    return matriz[i][j]

pesos = [1, 3, 2, 2]
valores = [1500, 2000, 500, 1000]
print(dp_mochila(5, pesos, valores))

3 1
0 arriba 1500
3 2
0 arriba 1500
2 1
0 arriba 1500
2 1
0 arriba 1500
3500


## Verificar si se puede llegar a un target con unas coins, se puede repetir coin.

In [ ]:
def verificar(coins, target, accum = 0):
    if accum == target:
        return True
    if accum > target:
        return False
    for coin in coins:
        resultado = verificar(coins, target, accum + coin)

    return resultado

verificar([1,3,5], 10)

### De cuantas maneras puedo llegar desde x,y hasta n-1, m-1



In [ ]:
matriz = [[1,1,1,1],
          [1,0,0,0],
          [1,0,0,0],
          [1,0,0,0]]

for i in range(1, len(matriz)):
    for j in range(1, len(matriz[0])): 
        izquierda = matriz[i][j - 1]
        arriba = matriz[i - 1][j]
        matriz[i][j] = izquierda + arriba

for fila in matriz:
    print(fila)


[1, 1, 1, 1]
[1, 2, 3, 4]
[1, 3, 6, 10]
[1, 4, 10, 20]


In [36]:
matriz = [[0,0,0,0],
          [0,0,0,0],
          [0,0,0,0],
          [0,0,0,0]]

x, y = 1,2
for i in range(x, len(matriz)):
    for j in range(y, len(matriz[0])):
        matriz[x][y] = 1
        print(i,j)

        izquierda = matriz[i][j - 1]
        arriba = matriz[i - 1][j]
        matriz[i][j] = izquierda + arriba

for fila in matriz:
    print(fila)


1 2
1 3
2 2
2 3
3 2
3 3
[0, 0, 0, 0]
[0, 0, 1, 1]
[0, 0, 1, 2]
[0, 0, 1, 3]


In [11]:
matriz = [[0,0,0,0],
          [0,0,"-",0],
          ["-",0,0,0],
          [0,0,0,0]]


def encontrar_maneras(matriz, x, y):
    print(x,y)
    if 0 > x >= len(matriz) or 0 > y >= len(matriz[0]):
        print(x,y)
        return 

    for i in range(x, len(matriz)):
        for j in range(y, len(matriz[0])):
            if matriz[i][j] == "-":
                continue
            if "-" in matriz[x] or "-" == matriz[i][y]:
                break

            matriz[x][y] = 1


            if matriz[i][j - 1] == "-":
                izquierda = 0
                arriba = matriz[i - 1][j]
                matriz[i][j] = izquierda + arriba
                continue

            if matriz[i - 1][j] == "-":
                arriba = 0
                izquierda = matriz[i][j - 1]
                matriz[i][j] = izquierda + arriba
                continue

            arriba = matriz[i - 1][j]
            izquierda = matriz[i][j - 1]
            matriz[i][j] = izquierda + arriba


    for fila in matriz:
        print(fila)

encontrar_maneras(matriz, 0, 0)

0 0
[1, 1, 1, 1]
[1, 2, '-', 1]
['-', 0, 0, 0]
[0, 0, 0, 0]
